# NSF — Arterial Pulse Tonometry: Signal Separation

## Project Overview

Non-invasive blood pressure monitoring via optical tonometry. A camera observes a deformable elastomer surface with printed fiducial markers (19×14 grid, 2mm spacing) placed over the radial artery. The surface deforms from two sources:

1. **Arterial pulse** — localized Gaussian spatial profile (σ ≈ 3mm), quasi-periodic at heart rate
2. **Motion artifacts** — spatially smooth (polynomial-like), from wrist movement

This project separates these two sources to extract clean pulse waveforms for BP estimation.

### Key Insight
The pulse is **spatially localized** (~4-8mm wide) while artifacts are **spatially global** (affect all markers). With 266 markers, >80% are artifact-only references, enabling spatial separation even when signals overlap spectrally.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
import sys
sys.path.insert(0, '..')

from src.synth.generator import GeneratorConfig, generate
from src.synth.pulse import PulseConfig
from src.synth.artifact import ArtifactConfig
from src.synth.noise import NoiseConfig
from src.separation.separator import SeparationConfig, separate
from src.separation.polynomial_fit import PolyFitConfig
from src.separation.metrics import evaluate, separation_snr
from src.viz.plots import plot_marker_grid, plot_displacement_timeseries, plot_artery_mask, plot_spatial_snapshot

## 1. Synthetic Data Generation

Generate a 10-second dataset at 30 fps with pulse, artifact, and noise.

In [ ]:
ds = generate(GeneratorConfig(
    num_frames=300, fps=30.0,
    pulse=PulseConfig(amplitude_mm=0.15, heart_rate_bpm=72.0, sigma_mm=3.0),
    artifact=ArtifactConfig(degree=2, amplitude_mm=1.0, seed=42),
    noise=NoiseConfig(sigma_mm=0.01, seed=43),
))

print(f'Grid: {ds.markers.grid_shape}, Frames: {ds.markers.num_frames}, Duration: {ds.markers.duration_sec:.1f}s')
print(f'Pulse RMS: {np.sqrt(np.mean(ds.ground_truth.pulse_displacement**2)):.4f} mm')
print(f'Artifact RMS: {np.sqrt(np.mean(ds.ground_truth.artifact_displacement**2)):.4f} mm')
print(f'Input SNR: {ds.separation_snr():.1f} dB')

In [ ]:
fig = plot_displacement_timeseries(ds)
plt.savefig('fig_displacement_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, comp, title in zip(axes, ['pulse', 'artifact', 'total'],
                            ['Pulse Only', 'Artifact Only', 'Total']):
    plot_spatial_snapshot(ds, frame=50, component=comp)
plt.savefig('fig_spatial_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Signal Separation

### Method: Artery-weighted polynomial fit
1. Compute displacements from rest position
2. Fit 2D polynomial (degree 2) to each frame, weighted by artery mask
3. Subtract polynomial → residual contains pulse + noise
4. (Optional) Project residual onto artery mask for Gaussian extraction

In [ ]:
gt = ds.ground_truth
gx, gy = gt.rest_positions[..., 0], gt.rest_positions[..., 1]

sep_cfg = SeparationConfig(
    polyfit=PolyFitConfig(degree=2),
    use_temporal_prefilter=False,
    use_gaussian_extraction=True,
)
result = separate(ds.markers, gx, gy, gt.artery_mask, sep_cfg)

pulse_rel = gt.pulse_displacement - gt.pulse_displacement[0:1]
artifact_rel = gt.artifact_displacement - gt.artifact_displacement[0:1]

m = evaluate(result.recovered_pulse, result.estimated_artifact,
             pulse_rel, artifact_rel, gt.artery_mask)
raw_snr = separation_snr(ds.markers.displacements_from_rest(), pulse_rel, artifact_rel)

print(f'SNR improvement: {m.separation_snr_db - raw_snr:.1f} dB')
print(f'Waveform correlation: {m.waveform_correlation:.3f}')
print(f'Artifact residual: {m.artifact_residual_fraction:.5f} ({m.artifact_residual_fraction*100:.3f}%)')

## 3. Baseline Sweep Results (Phase 2)

Performance across 6 validation axes on polynomial artifacts.

In [ ]:
with open('../experiments/baseline_results.json') as f:
    baseline = json.load(f)

# Group by axis
axes_data = {}
for r in baseline:
    axis = r['axis']
    if axis not in axes_data:
        axes_data[axis] = {'values': [], 'snr': [], 'wfm': [], 'art': []}
    axes_data[axis]['values'].append(r['value'])
    axes_data[axis]['snr'].append(r['snr_improvement_db'])
    axes_data[axis]['wfm'].append(r['waveform_correlation'])
    axes_data[axis]['art'].append(r['artifact_residual_fraction'])

fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, data) in zip(axs.flat, axes_data.items()):
    ax.plot(data['values'], data['snr'], 'bo-')
    ax.set_xlabel(name)
    ax.set_ylabel('SNR improvement (dB)')
    ax.set_title(name.replace('_', ' ').title())
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_baseline_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Physics-Based Artifacts (Phase 3)

Performance gap between polynomial and physics-based artifacts.

In [ ]:
with open('../experiments/physics_artifact_results.json') as f:
    physics = json.load(f)

print('Method Comparison on Physics-Based Artifacts')
print('=' * 65)
for method in ['poly_only', 'poly_plus_gaussian', 'iterative_3', 'joint_model']:
    if method in physics:
        valid = [r for r in physics[method] if 'error' not in r]
        if valid:
            snrs = [r['snr_improvement_db'] for r in valid]
            wfms = [r['waveform_correlation'] for r in valid]
            arts = [r['artifact_residual_fraction'] for r in valid]
            print(f'{method:25s}  SNR={np.mean(snrs):5.1f}±{np.std(snrs):4.1f}dB  '
                  f'WFM={np.mean(wfms):6.3f}  ART={np.mean(arts):5.3f}')

## 5. Architecture

```
MarkerTimeSeries (T, R, C, 2)
    │
    ▼
┌─────────────────────────────┐
│  Signal Separation Pipeline │
│                             │
│  1. Displacements from rest │
│  2. Polynomial artifact fit │
│  3. Artifact subtraction    │
│  4. Gaussian pulse extract  │
└─────────────────────────────┘
    │
    ▼
┌─────────────────────────────┐
│  Pulse Extraction           │
│                             │
│  • SNR-weighted averaging   │
│  • Bandpass filtering       │
│  • HR detection (FFT)       │
└─────────────────────────────┘
    │
    ▼
┌─────────────────────────────┐
│  BP Estimation              │
│                             │
│  • Beat segmentation        │
│  • Morphology extraction    │
│  • Calibrated regression    │
└─────────────────────────────┘
    │
    ▼
BPEstimate (systolic, diastolic, MAP)
```

### Module Map
```
src/
├── data/          MarkerTimeSeries, loader, simulator_bridge
├── synth/         pulse, artifact, noise, generator
├── separation/    temporal_filter, polynomial_fit, separator,
│                  metrics, gaussian_extractor, joint_model,
│                  decomposition, subspace_separation
├── estimation/    pulse_extractor, spatial_fit, artifact_stats,
│                  param_library, bp_estimation
├── viz/           animate, plots
└── pipeline.py    End-to-end + real-time separator
```

## 6. Key Results Summary

| Metric | Polynomial Artifacts | Physics-Based Artifacts | Spec Target |
|--------|---------------------|------------------------|-------------|
| SNR improvement | **27 dB** | **24.5 dB** (hybrid) | ≥ 10 dB |
| Waveform correlation | **0.984** | **0.32** (variable) | ≥ 0.95 |
| Artifact residual | **0.007%** | **65%** | <5% / <15% |
| Real-time latency | **<1 ms/frame** | — | <5 ms |
| Tests | **102 passing** | — | — |

### Key Findings
1. **Spatial separation works**: The localized-vs-global spatial structure enables 27 dB SNR improvement on polynomial artifacts
2. **Physics-based artifacts are hard**: Curl and elastic noise aren't polynomial → 65% residual
3. **Artery mask alignment dominates**: Well-aligned scenarios achieve 0.96 correlation; poorly aligned → negative
4. **Hybrid poly+subspace is best**: Polynomial captures global component, subspace captures non-polynomial remainder
5. **Real-time is feasible**: <1ms per frame with precomputed matrices